# 4.8 增加深度改變了什麼？

# 第 4 章：第一個 Dense Transformer 語言模型

前置：第3章的因果 attention。完整可讀實作在 tiny_perceptron/model.py；預設只用 learned position、LayerNorm、Dense FFN，還沒有 FlashAttention 或 MoE。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像原稿旁加修訂，而不是撕掉原稿**

y=x+f(x)，原資料和修訂都有路徑。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/residual.svg")))

**先想一想：**兩層模型的總參數剛好是單層兩倍嗎？

增加block讓表示能多次混合與修正，也增加參數和計算。未訓練模型的隨機loss不能證明更深一定更好。

**把直覺寫成數學：**深度增加主要block參數；embedding與輸出層不會跟著每層複製。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

for depth in (1, 2, 3):
    model = TinyLM(ModelConfig(vocab_size=20, width=8, layers=depth))
    print(depth, model.description()["parameters"])
    assert model(torch.tensor([[1, 2]]))["logits"].shape == (1, 2, 20)

**如何讀結果：**比較品質要另外用相同資料與明示預算訓練，這裡只檢查結構。

**只改一件事：**只改width，觀察參數不是線性增長。
